# 6.3 — 照合して再現可能にする


小さなfixture、全行照合、保存後の再読込を、一つの独立した確認手順にします。


In [ ]:
from pathlib import Path
import pandas as pd

project = Path.cwd() / "projects" / "clinic-stock-scaleup"
fixture = project / "data" / "clinic-stock-fixture.csv"
print("Project found:", project.is_dir())
print("Fixture found:", fixture.is_file())


## 6.3.1 原本を人間が確認できる形で見る


In [ ]:
records = pd.read_csv(fixture)
print("Rows:", len(records), "Columns:", len(records.columns))
print(records.to_string(index=False))


## 6.3.2 件数を照合する


In [ ]:
required_numeric = ["opening_units", "received_units", "dispensed_units", "closing_units", "stockout_hours", "patients_turned_away"]
work = records.copy(deep=True)
for column in required_numeric:
    work[column] = pd.to_numeric(work[column], errors="coerce")
invalid = work[required_numeric].isna().any(axis=1) | work[required_numeric].lt(0).any(axis=1)
analysis = work.loc[~invalid].copy()
review = work.loc[invalid].copy()
print("Source:", len(work), "Analysis:", len(analysis), "Review:", len(review))
assert len(work) == len(analysis) + len(review)


## 6.3.3 保存結果を再読込する


In [ ]:
output = project / "output" / "lesson63_check.csv"
output.parent.mkdir(exist_ok=True)
check_table = analysis.groupby(["district", "medicine"], as_index=False).agg(records=("date", "size"))
check_table.to_csv(output, index=False)
saved = pd.read_csv(output)
assert list(saved.columns) == list(check_table.columns)
assert len(saved) == len(check_table)
display(saved)


## 確認
この確認が保証することと、まだ保証しないことを一つずつ挙げてください。
